# Chapter 5 — Building Personal Assistants: Chains (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare)

**Learning objectives**
- Build history-aware retrieval chains
- Compose chains with LCEL (prompt | model | parser)
- Run parallel and sequential chains for scientific/medical tasks
- Build debate and custom-function chains

> Runtime: ~10 min (API)  
> Cost: paid LLM required  
> Data: synthetic biology/agriculture/medicine examples

> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.



This notebook demonstrates how to build personal assistants using LangChain's chain capabilities. We'll explore various chain types including history-aware retrievers, sequential chains, and parallel chains for different use cases in life sciences and research.

## Key Takeaways

- **Chain Composition**: Use LCEL for clean, readable chain building
- **Context Awareness**: History-aware retrievers improve conversational AI
- **Parallel Processing**: Optimize performance with simultaneous operations
- **Modularity**: Break complex tasks into smaller, manageable chains
- **Testing**: Always test chains with various inputs to ensure robustness
- **Visualization**: Use `get_graph().print_ascii()` to understand chain structure

## Package Installation

In [1]:
# @title Installing Python dependencies
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "langchain-community==0.4.0" "langchain-text-splitters==1.0.0" "langgraph>=0.2" "grandalf>=0.8" "docarray>=0.40" "faiss-cpu>=1.8" "pandas>=2.0" "matplotlib>=3.8" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)

In [2]:
import importlib.metadata as _md

for _p in [
    "langchain",
    "langchain-core",
    "langchain-openai",
    "langchain-community",
    "langgraph",
]:
    try:
        print(_p, _md.version(_p))
    except Exception:
        print(_p, "not installed")

langchain 1.0.0
langchain-core 1.2.30
langchain-openai 1.0.0
langchain-community 0.4
langgraph 1.0.10


In [3]:
# @title Installing Python dependencies
!pip install torch --index-url https://download.pytorch.org/whl/cpu

Found existing installation: torch 2.13.0+cpu
Uninstalling torch-2.13.0+cpu:
  Successfully uninstalled torch-2.13.0+cpu
Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached https://download-r2.pytorch.org/whl/cpu/torch-2.13.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (37 kB)
Using cached https://download-r2.pytorch.org/whl/cpu/torch-2.13.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl (191.8 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.28 requires torchvision, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.


In [4]:
# @title Setting environmental variables
import os

# --- Dual-mode secrets: works in Google Colab AND locally (.env / environment) ---
try:
    from google.colab import userdata  # type: ignore

    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False

if not IN_COLAB:
    # Local run: load variables from a .env file if present (never commit .env!).
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except Exception:
        pass


def get_secret(name, default=None):
    """Read a secret from Colab Secrets, else from local env/.env, else default."""
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)


# 👇 Choose your provider 👇
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"

if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret(
        "LC4LSH_ANTHROPIC_API_KEY", "sk-ant-..."
    )
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")

print(
    f"✅ API keys loaded for {API_KEY_PROVIDER} (source: {'Colab Secrets' if IN_COLAB else 'local env/.env'})"
)

# Hugging Face token (optional; needed for gated models)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""

API keys loaded for OPENAI


In [5]:
# @title Setting LangSmith variables
# ========================
# 👇 CONFIGURE HERE 👇
# ========================
LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY") or "lsv2_pt_..."
LANGSMITH_PROJECT = "lc4lsh-chapter5-chains"  # Traces appear under this name
REGION = "EU"  # "EU" or "US" - must match your account!
# ========================

if LANGSMITH_API_KEY and LANGSMITH_API_KEY != "lsv2_pt_...":
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    os.environ["LANGSMITH_ENDPOINT"] = (
        "https://eu.api.smith.langchain.com"
        if REGION == "EU"
        else "https://api.smith.langchain.com"
    )
    dashboard = (
        "https://eu.smith.langchain.com"
        if REGION == "EU"
        else "https://smith.langchain.com"
    )
    print(f"✅ LangSmith enabled!")
    print(f"   Region: {REGION} | Project: {LANGSMITH_PROJECT}")
    print(f"   Dashboard: {dashboard}")
else:
    print("⚠️ LangSmith disabled - paste your API key above to enable tracing")
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF")

# Back-compat env vars some older cells reference
os.environ.setdefault(
    "LANGCHAIN_TRACING_V2", os.environ.get("LANGSMITH_TRACING", "false")
)
os.environ.setdefault("LANGCHAIN_PROJECT", LANGSMITH_PROJECT)

✅ LangSmith enabled!
   Region: EU | Project: lc4lsh-chapter5-chains
   Dashboard: https://eu.smith.langchain.com


'lc4lsh-chapter5-chains'

## History-Aware Retrieval Chain

### What is a History-Aware Retrieval Chain?
A history-aware retrieval chain is a sophisticated RAG (Retrieval-Augmented Generation) system that considers conversation history when retrieving relevant documents. This allows the system to understand context from previous messages and provide more accurate responses.


In [6]:
from langchain_core.prompts import MessagesPlaceholder, ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import create_history_aware_retriever
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Initialize LLM and embedding model
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

# Sample documents (replace with your actual data)
documents = [
    Document(
        page_content="Experiment A: We tested the effect of fertilizer X on tomato yield at temperature 80°F. Results showed a 20% increase in yield in the experimental group."
    ),
    Document(
        page_content="Experiment B: We tested the effect of fertilizer X on corn yield. The experiment is ongoing."
    ),
    Document(
        page_content="Experiment C: We tested the effect of compost and different temperatures on strawberry growth. Best results achieved at 25°C-30°C range."
    ),
    Document(
        page_content="Experiment D: Examined the small influence of compost amount on soil acidity."
    ),
    Document(
        page_content="Experiment E: Examined the influence of light intensity on photosynthesis in algae."
    ),
]

# Create vector store
vector = FAISS.from_documents(documents, embedding_model)
retriever = vector.as_retriever(k=3)

# Prompt for history-aware retrieval
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You help users find information about agricultural experiments.
                Paraphrase the user's query based on the conversation history.""",
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("user", "{input}"),
    ]
)

# Create history-aware retriever chain
history_aware_retriever_chain = create_history_aware_retriever(llm, retriever, prompt)

### Test 1: Query with Context Updates

In [7]:
chat_history = [
    HumanMessage(content="Experiment E actually used Fertilizer X."),
    AIMessage(
        content="Thank you for the correction. I'll note that Experiment E used Fertilizer X."
    ),
    HumanMessage(
        content="Experiment B had contamination issues and should be excluded from analysis."
    ),
    AIMessage(
        content="Noted. I'll exclude Experiment B from all analysis due to contamination issues."
    ),
]

In [8]:
query = "List all experiment where fertilizer X was used"

# Invoke the retriever chain
result = history_aware_retriever_chain.invoke(
    {"chat_history": chat_history, "input": query}
)


**Expected Result**: The system should retrieve documents related to fertilizer X, considering the conversation history that Experiment E used Fertilizer X and Experiment B should be excluded.


In [9]:
result

[Document(id='691a868c-68a5-468e-9880-204b47774e25', metadata={}, page_content='Experiment B: We tested the effect of fertilizer X on corn yield. The experiment is ongoing.'),
 Document(id='9487675f-5b13-423d-bde3-11b338a40add', metadata={}, page_content='Experiment A: We tested the effect of fertilizer X on tomato yield at temperature 80°F. Results showed a 20% increase in yield in the experimental group.'),
 Document(id='31a5461e-4308-4862-b13d-fd15ae675c16', metadata={}, page_content='Experiment D: Examined the small influence of compost amount on soil acidity.'),
 Document(id='820c9e68-8864-4974-b290-7b52fc7412ae', metadata={}, page_content='Experiment E: Examined the influence of light intensity on photosynthesis in algae.')]

In [10]:
system_prompt = """You are an assistant for question-answering tasks.
    Use the following pieces of retrieved context to answer
    the question. If you don't know the answer, say that you
    don't know. Always use the metric system to answer questions!
    {context}"""

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever_chain, question_answer_chain)

In [11]:
result = rag_chain.invoke({"input": query, "chat_history": chat_history})

In [12]:
result

{'input': 'List all experiment where fertilizer X was used',
 'chat_history': [HumanMessage(content='Experiment E actually used Fertilizer X.', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Thank you for the correction. I'll note that Experiment E used Fertilizer X.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Experiment B had contamination issues and should be excluded from analysis.', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Noted. I'll exclude Experiment B from all analysis due to contamination issues.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'context': [Document(id='691a868c-68a5-468e-9880-204b47774e25', metadata={}, page_content='Experiment B: We tested the effect of fertilizer X on corn yield. The experiment is ongoing.'),
  Document(id='9487675f-5b13-423d-bde3-11b338a40add', metadata={}, page_content='Experiment A: We tested t

### Test 2: Query Without Initial Context

In [13]:
query = "List all experiment where Fertilizer Y was used"

# Invoke the retriever chain
result = history_aware_retriever_chain.invoke({"chat_history": [], "input": query})

**Expected Result**: The system should not find any experiments with Fertilizer Y since none are mentioned in the documents.

In [14]:
result

[Document(id='691a868c-68a5-468e-9880-204b47774e25', metadata={}, page_content='Experiment B: We tested the effect of fertilizer X on corn yield. The experiment is ongoing.'),
 Document(id='9487675f-5b13-423d-bde3-11b338a40add', metadata={}, page_content='Experiment A: We tested the effect of fertilizer X on tomato yield at temperature 80°F. Results showed a 20% increase in yield in the experimental group.'),
 Document(id='31a5461e-4308-4862-b13d-fd15ae675c16', metadata={}, page_content='Experiment D: Examined the small influence of compost amount on soil acidity.'),
 Document(id='820c9e68-8864-4974-b290-7b52fc7412ae', metadata={}, page_content='Experiment E: Examined the influence of light intensity on photosynthesis in algae.')]

### Test 3: Query with Semantic Understanding

In [15]:
chat_history = [
    HumanMessage(content="Was Fertilizer Y used in the experiemnts.?"),
    AIMessage(content="No, Fertilizer Y wasn't used in any of the experiments."),
    HumanMessage(content="That's incorrect. Fertilizer Y and compost are the same."),
    AIMessage(
        content="Thank you for the correction. I'll note that Fertilizer Y and compost are the same."
    ),
]

In [16]:
query = "List all experiment where Fertilizer Y was used"

# Create history-aware retriever chain
history_aware_retriever_chain = create_history_aware_retriever(llm, retriever, prompt)

# Invoke the retriever chain
result = history_aware_retriever_chain.invoke(
    {"chat_history": chat_history, "input": query}
)

In [17]:
result

[Document(id='691a868c-68a5-468e-9880-204b47774e25', metadata={}, page_content='Experiment B: We tested the effect of fertilizer X on corn yield. The experiment is ongoing.'),
 Document(id='31a5461e-4308-4862-b13d-fd15ae675c16', metadata={}, page_content='Experiment D: Examined the small influence of compost amount on soil acidity.'),
 Document(id='9487675f-5b13-423d-bde3-11b338a40add', metadata={}, page_content='Experiment A: We tested the effect of fertilizer X on tomato yield at temperature 80°F. Results showed a 20% increase in yield in the experimental group.'),
 Document(id='4fc29535-cc60-44fd-b92f-5bfe67251dcd', metadata={}, page_content='Experiment C: We tested the effect of compost and different temperatures on strawberry growth. Best results achieved at 25°C-30°C range.')]

In [18]:
system_prompt = """You are an assistant for question-answering tasks.
    Use the following pieces of retrieved context to answer
    the question. If you don't know the answer, say that you
    don't know. Always use the metric system to answer questions!
    {context}"""

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever_chain, question_answer_chain)

In [19]:
result = rag_chain.invoke({"input": query, "chat_history": chat_history})


**Expected Result**: The system should now understand that Fertilizer Y = compost and retrieve experiments C and D that used compost.


In [20]:
result

{'input': 'List all experiment where Fertilizer Y was used',
 'chat_history': [HumanMessage(content='Was Fertilizer Y used in the experiemnts.?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="No, Fertilizer Y wasn't used in any of the experiments.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content="That's incorrect. Fertilizer Y and compost are the same.", additional_kwargs={}, response_metadata={}),
  AIMessage(content="Thank you for the correction. I'll note that Fertilizer Y and compost are the same.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'context': [Document(id='691a868c-68a5-468e-9880-204b47774e25', metadata={}, page_content='Experiment B: We tested the effect of fertilizer X on corn yield. The experiment is ongoing.'),
  Document(id='31a5461e-4308-4862-b13d-fd15ae675c16', metadata={}, page_content='Experiment D: Examined the small influence of compost amoun

## Basic Chain Building with LCEL

### What is LCEL?
LangChain Expression Language (LCEL) is a declarative way to compose chains. It provides a simple syntax for building complex workflows using the pipe operator (|).


In [21]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [22]:
model = ChatOpenAI(model="gpt-5-nano")
prompt = ChatPromptTemplate.from_template(
    "You are a chemist. Answer the following question: {question}"
)
output_parser = StrOutputParser()

### Chain 1: Direct Model Invocation

In [23]:
chain_1 = model
chain_1.invoke(("human", "What is a bond?"))

AIMessage(content='There are two common meanings for "bond." Here are quick, human-friendly definitions of each:\n\n1) Financial bond (debt instrument)\n- What it is: A loan you make to an issuer (like a government or a company). In return, the issuer promises to pay you regular interest (a coupon) and to repay the principal (the par value) at a set date (the maturity).\n- How it works in simple terms: You buy the bond for some price. Over time you receive periodic coupon payments. At maturity you get back the face value (often $1,000). The bond’s price can move up or down before maturity depending on interest rates and the issuer’s credit.\n- Why people buy bonds: They provide predictable income and can be less volatile than stocks, helping with diversification and capital preservation.\n- Key features to know: \n  - Par value (face value), coupon rate (interest rate stated on the bond), coupon payments (often semiannual), maturity date, and price.\n  - Yield: the return you’d earn if

### Chain 2: Prompt + Model

In [24]:
chain_2 = prompt | model
chain_2.invoke({"question": "What is a bond?"})

AIMessage(content='A bond is the force that holds atoms together in a molecule or solid. It arises from interactions that lower the overall energy of the system, so that the atoms are more stable together than apart.\n\nKey ways bonds form and what they mean:\n\n- Covalent bonds: atoms share one or more pairs of electrons. This fills valence shells and creates a stable unit (for example, H2 or O2). Bond order (single, double, triple) and bond length determine strength and distance between nuclei.\n- Ionic bonds: electrons are transferred from one atom to another, creating positively and negatively charged ions that attract each other (e.g., NaCl). This often occurs between a metal and a nonmetal with a large electronegativity difference.\n- Metallic bonds: a lattice of positive metal ions with a “sea” of delocalized electrons. This accounts for conductivity and malleability of metals.\n- Hydrogen bonds and other intermolecular interactions: stronger dipole-dipole interactions (often in

### Chain 3: Complete Chain with Output Parser

In [25]:
chain_3 = prompt | model | output_parser
chain_3.invoke({"question": "What is a bond?"})

'A bond is a stable interaction that holds two or more atoms together in a molecule or solid. It arises because electrons and nuclei interact in ways that lower the system’s overall energy, so the bonded arrangement is more favorable than the separate atoms.\n\nKey ideas:\n- Types of bonds:\n  - Covalent: atoms share electrons (e.g., H2, O2, CH4).\n  - Ionic: electrons are transferred, creating charged ions that attract (e.g., NaCl).\n  - Metallic: electrons are delocalized over a lattice of atoms (e.g., Fe, Cu).\n  - Secondary or weaker interactions: hydrogen bonds, dipole-dipole, London dispersion forces (van der Waals).\n- Primary vs secondary: Primary bonds (covalent/ionic/metallic) hold atoms in a molecule or lattice; secondary interactions help determine structure and properties but are weaker.\n- Characteristics:\n  - Bond length: the distance between nuclei at the lowest energy.\n  - Bond energy (strength): energy required to break the bond.\n  - Bond order: single, double, tri


**Key Difference**: Chain 3 returns a clean string instead of a message object, making it easier to work with programmatically.


## RAG Chain with Parallel Processing

### What is Parallel Processing in Chains?
Parallel processing allows multiple operations to run simultaneously, improving efficiency. In RAG systems, we can retrieve context and pass through the question in parallel.


### Setup Biology Knowledge Base

In [26]:
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

# Initialize the model
model = ChatOpenAI(model="gpt-5-nano")

vectorstore = DocArrayInMemorySearch.from_texts(
    [
        "DNA carries genetic information within cell chromosomes.",
        "Ecosystems consist of living organisms and their physical environment.",
        "The human heart pumps blood through arteries and veins.",
        "Some bacteria cause diseases while others are beneficial.",
        "Homeostasis maintains steady internal conditions in living systems.",
        "Natural selection helps organisms adapt and survive in their environments.",
        "Mitochondria produce ATP, the main energy source for cells.",
        "Photosynthesis in plants produces oxygen.",
        "The brain controls body functions and is located in the skull.",
        "The immune system defends against harmful substances by detecting antigens.",
    ],
    embedding=OpenAIEmbeddings(model="text-embedding-3-large"),
)
retriever = vectorstore.as_retriever()

### Create RAG Chain with Parallel Processing

In [27]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
output_parser = StrOutputParser()

# Setup retrieval and handling of inputs
retrieval = RunnableParallel({"context": retriever, "question": RunnablePassthrough()})

# Combine the components into a processing chain
chain = retrieval | prompt | model | output_parser

### Test Biology RAG Chain

In [28]:
result1 = chain.invoke("How do plants release oxygen?")
print(result1)

During photosynthesis.


In [29]:
result2 = chain.invoke("What is the main energy source for cells?")
print(result2)

ATP (adenosine triphosphate).


In [30]:
result3 = chain.invoke("Who painted mona Lisa?")
print(result3)

The provided documents do not mention who painted the Mona Lisa.


**Expected Results**:
- Test 1: Should reference photosynthesis
- Test 2: Should reference ATP and mitochondria
- Test 3: Should indicate the answer is not in the context

## Sequential Chains for Medical Diagnosis

### What are Sequential Chains?
Sequential chains process information in steps, where the output of one chain becomes the input for the next. This is useful for multi-step reasoning tasks.


### Medical Diagnosis Chain

In [31]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Prompt to determine the disease based on symptoms
prompt_symptom = ChatPromptTemplate.from_template(
    "Based on these symptoms, what disease might this person have: {symptoms}?"
)

# Prompt to recommend lab exams based on the suspected disease
prompt_disease = ChatPromptTemplate.from_template(
    "What lab exams should be taken to confirm a diagnosis of {disease}?"
)

model = ChatOpenAI()

# Chains to determine the disease from symptoms
sub_chain_symptom = prompt_symptom | model | StrOutputParser()
sub_chain_disease = prompt_disease | model | StrOutputParser()

# Sequential chain: symptoms -> disease -> lab tests
main_chain = {"disease": sub_chain_symptom} | sub_chain_disease

### Visualize Chain Structure

In [32]:
main_chain.get_graph().print_ascii()

+------------------------+ 
| Parallel<disease>Input | 
+------------------------+ 
             *             
             *             
             *             
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
             *             
             *             
             *             
      +------------+       
      | ChatOpenAI |       
      +------------+       
             *             
             *             
             *             
    +-----------------+    
    | StrOutputParser |    
    +-----------------+    
             *             
             *             
             *             
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
             *             
             *             
             *             
      +------------+       
      | ChatOpenAI |       
      +------------+       
             *             
             *             
             *      

### Test Medical Diagnosis Chain

In [33]:
# Example invocation to test the chain
result = main_chain.invoke({"symptoms": "fever, cough, and shortness of breath"})
print("Recommended Lab Exams:", result)

Recommended Lab Exams: The lab exams that should be taken to confirm a diagnosis of COVID-19 include:

1. Polymerase chain reaction (PCR) test: This is the most common test used to diagnose COVID-19. It detects the genetic material of the virus in a respiratory sample collected from the patient, usually a nasopharyngeal swab.

2. Antigen test: This test can also detect viral proteins in a respiratory sample and is often used as a rapid test for COVID-19 diagnosis.

3. Antibody test: This blood test can detect the presence of antibodies produced by the immune system in response to a COVID-19 infection. It is usually used to determine if a person has previously been infected with the virus.

It is important for individuals experiencing symptoms of COVID-19 to seek medical attention and follow the guidance of healthcare providers regarding testing and treatment. It is also essential to follow public health guidelines to prevent the spread of the virus to others.



**Expected Result**: The system should first identify potential diseases (like COVID-19, pneumonia) and then recommend appropriate lab tests (PCR, chest X-ray, blood tests).


## Complex Parallel Chain for Scientific Research

### Scientific Hypothesis Testing Chain
This chain demonstrates how to create complex workflows that branch and merge information.


In [34]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI

model = ChatOpenAI()

# Prompt for generating a hypothesis based on an observation
prompt_observation = ChatPromptTemplate.from_template(
    "Based on the observation: {observation}, what is a possible biological explanation or hypothesis?"
)

# Prompt for suggesting an experiment to test a hypothesis
prompt_hypothesis = ChatPromptTemplate.from_template(
    "What experiment could we perform to test the hypothesis: {hypothesis}, considering the condition: {condition}?"
)

# Prompt for predicting the outcome of an experiment
prompt_experiment = ChatPromptTemplate.from_template(
    "Given the setup: {experiment_setup}, what might be the expected outcome of this experiment?"
)

### Build Multi-Step Research Chain

In [35]:
# Chain to generate a hypothesis from an observation
hypothesis_generator = (
    {"observation": RunnablePassthrough()}
    | prompt_observation
    | model
    | StrOutputParser()
)

# Chain to suggest an experiment based on the hypothesis, merging with additional input
experiment_suggestion = (
    RunnableParallel(
        {
            "hypothesis": hypothesis_generator,
            "condition": RunnablePassthrough(),
        }  # Merging the hypothesis output with a new input 'condition'
    )
    | prompt_hypothesis
    | model
    | StrOutputParser()
)

# Chain to predict the outcome of the suggested experiment
experiment_outcome = (
    {"experiment_setup": experiment_suggestion}
    | prompt_experiment
    | model
    | StrOutputParser()
)

### Visualize Research Chain

In [36]:
experiment_outcome.get_graph().print_ascii()

           +---------------------------------+         
           | Parallel<experiment_setup>Input |         
           +---------------------------------+         
                            *                          
                            *                          
                            *                          
         +-------------------------------------+       
         | Parallel<hypothesis,condition>Input |       
         +-------------------------------------+       
                   **               ***                
                ***                    ***             
              **                          ***          
    +-------------+                          **        
    | Passthrough |                           *        
    +-------------+                           *        
           *                                  *        
           *                                  *        
           *                                  * 

### Test Scientific Research Chain

In [37]:
# Example invocation to test the chain
result = experiment_outcome.invoke(
    {
        "observation": "Pea plants with round seeds produce mostly round seed offspring, even when crossed with wrinkled seeds.",
        "condition": "controlled pollination",
    }
)
print("Experiment Prediction:", result)

Experiment Prediction: The expected outcome of this experiment would be that a majority of the offspring plants would have round seeds. This would indicate that the gene for round seed shape is dominant over the gene for wrinkled seed shape in pea plants. If the hypothesis is correct, the offspring generation would predominantly exhibit the round seed shape trait, even when the parent plants had a mix of round and wrinkled seed shapes. This would support the idea that the gene for round seed shape is dominant in pea plants.


**Expected Result**: The system should generate a hypothesis about dominant/recessive traits, suggest controlled breeding experiments, and predict outcomes based on genetic principles.


## Debate Chain with Argument Analysis

### Creating a Balanced Debate System
This chain demonstrates how to create a system that can analyze arguments from multiple perspectives.

In [38]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

# Initialize the model
model = ChatOpenAI()

# Prompt to generate an initial argument about a controversial life science topic
generate_argument = (
    ChatPromptTemplate.from_template("Generate an argument about: {input}")
    | model
    | StrOutputParser()
    | {"base_argument": RunnablePassthrough()}
)

# Chain to list the positive aspects (pros) of the argument
arguments_for = (
    ChatPromptTemplate.from_template(
        "List the pros or positive aspects of: {base_argument}"
    )
    | model
    | StrOutputParser()
)

# Chain to list the negative aspects (cons) of the argument
arguments_against = (
    ChatPromptTemplate.from_template(
        "List the cons or negative aspects of: {base_argument}"
    )
    | model
    | StrOutputParser()
)

# Template for the final responder to synthesize the debate
final_responder = (
    ChatPromptTemplate.from_template(
        "Discussion on {input}:\n\nPros:\n{arguments_for}\n\nCons:\n{arguments_against}\n\nCan you provide a balanced conclusion?"
    )
    | model
    | StrOutputParser()
)

### Complete Debate Chain

In [39]:
# Full chain to manage the debate
main_chain = (
    generate_argument
    | {
        "arguments_for": arguments_for,
        "arguments_against": arguments_against,
        "input": itemgetter("base_argument"),
    }
    | final_responder
)

### Visualize Debate Chain

In [40]:
main_chain.get_graph().print_ascii()

                                   +-------------+                               
                                   | PromptInput |                               
                                   +-------------+                               
                                          *                                      
                                          *                                      
                                          *                                      
                               +--------------------+                            
                               | ChatPromptTemplate |                            
                               +--------------------+                            
                                          *                                      
                                          *                                      
                                          *                                      
                

### Test Debate Chain

In [41]:
result = main_chain.invoke({"input": "the use of CRISPR technology in human embryos"})
print("Debate Summary:", result)

Debate Summary: In a balanced conclusion, it is important to acknowledge that the use of CRISPR technology in human embryos presents both potential benefits and risks. While it has the potential to cure genetic diseases, improve overall health, and address infertility issues, it also raises concerns about inequality, discrimination, ethical considerations, and the unknown long-term effects on future generations. 

It is crucial for society to carefully consider the ethical implications, regulatory oversight, and responsible use of this technology in order to ensure that it is used to advance human health and well-being while also respecting individual autonomy and rights. As research and technology continue to progress, it is essential that these considerations remain at the forefront of any discussions and decisions regarding the use of CRISPR technology in human embryos.


**Expected Result**: The system should generate a balanced analysis covering benefits (treating genetic diseases) and concerns (ethical implications, safety) of CRISPR technology.



## Part 7: Custom Chain Functions

### What are Custom Chain Functions?
Custom chain functions allow you to create reusable chain components with the `@chain` decorator, providing more flexibility than standard LCEL chains.



### Drug Discovery Analysis Chain

In [42]:
from langchain_core.runnables import chain

generate_scenario_prompt = ChatPromptTemplate.from_template(
    "Generate a hypothetical drug discovery scenario involving {topic}"
)
analyze_scenario_prompt = ChatPromptTemplate.from_template(
    "What are the key challenges and potential solutions in this scenario: {scenario}"
)


@chain
def drug_discovery_analysis(topic):
    # Generate a scenario based on the given drug discovery topic
    initial_scenario = generate_scenario_prompt.invoke({"topic": topic})
    scenario_description = ChatOpenAI().invoke(initial_scenario)
    parsed_scenario = StrOutputParser().invoke(scenario_description)

    # Analyze the generated scenario to identify challenges and potential solutions
    analysis_chain = analyze_scenario_prompt | ChatOpenAI() | StrOutputParser()
    scenario_analysis = analysis_chain.invoke({"scenario": parsed_scenario})

    return scenario_analysis

In [43]:
topic_focus = "Alzheimer's disease"
scenario_analysis_result = drug_discovery_analysis.invoke(topic_focus)
print("Analysis of Drug Discovery Scenario:", scenario_analysis_result)

Analysis of Drug Discovery Scenario: Key Challenges:
1. Despite promising results in preclinical and Phase I trials, there is always a risk that the drug may not demonstrate the same level of effectiveness in larger Phase II and III trials, leading to potential setbacks in the drug development process.
2. Alzheimer's disease is a complex and multifaceted condition with no known cure, making it challenging to develop a treatment that effectively targets its underlying mechanisms and provides meaningful benefits for patients.
3. The high cost and time-consuming nature of clinical trials for Alzheimer's treatments can present financial and logistical challenges for pharmaceutical companies, especially if the drug does not ultimately receive regulatory approval.

Potential Solutions:
1. Prioritize robust and comprehensive preclinical research to better understand the underlying mechanisms of Alzheimer's disease and identify potential drug targets with a higher likelihood of success in huma


**Expected Result**: The system should generate a realistic drug discovery scenario for Alzheimer's disease and analyze key challenges like blood-brain barrier penetration, clinical trial design, and regulatory approval.

## Summary

This notebook demonstrated several key concepts in building personal assistants with LangChain:

1. **History-Aware Retrieval**: Building systems that understand conversation context
2. **LCEL Chains**: Using the pipe operator for clean, declarative chain building
3. **Parallel Processing**: Improving efficiency with simultaneous operations
4. **Sequential Chains**: Multi-step reasoning for complex tasks
5. **Complex Workflows**: Branching and merging information streams
6. **Custom Chain Functions**: Creating reusable chain components

These patterns form the foundation for building sophisticated AI assistants that can handle complex, multi-step reasoning tasks while maintaining context and providing accurate, relevant responses.

## Limitations & safety notes

- **Medical diagnosis chains are illustrative only** — not for clinical use.
- **LCEL chains are not validated reasoning**; the model can still hallucinate between steps.
- **Paid API required** for all `ChatOpenAI` cells.
- `create_*_chain` helpers now live in `langchain_classic` under LangChain 1.0.


---
### Further Reading

| Notebook | Relevance |
|----------|-----------|
| **Chapter 3 Simple Pipeline-4** | LCEL patterns introduced in Ch3 |
| **Chapter 5 Building Personal Assistants LangGraph and Agents** | Graduating from chains to agents |


In [44]:
# Cleanup
import gc

for _v in ("vector", "retriever", "vectorstore", "llm", "model", "main_chain"):
    globals().pop(_v, None)
gc.collect()
print("Cleanup complete.")

Cleanup complete.


## Exercises

<details><summary>Why use a history-aware retriever?</summary>It rewrites follow-up questions into standalone queries using chat history, so retrieval works with pronouns/context.</details>

<details><summary>What does the `|` operator do in LCEL?</summary>It pipes runnables so the output of one becomes the input of the next (prompt | model | parser).</details>

<details><summary>Sequential vs parallel chains?</summary>Sequential feeds step N's output into N+1; parallel runs branches concurrently and merges results.</details>

### Tasks
- **Task A** - Convert the medical diagnosis chain to add a third step that suggests specialist referral.
- **Task B** - Add a citation step to the RAG chain that returns which document supported the answer.
- **Task C** - Swap `gpt-...` for a local Ollama model in one chain and compare outputs.
- **Task D** - Add a `StrOutputParser` + JSON parser variant of the debate chain that returns structured pros/cons.
